# EcoVision AI — final training on Google Colab

This notebook trains **both** models on the full RealWaste dataset, evaluates them once on the held-out test split, and packages the artifacts the local app expects.

**What you get at the end:** `ecovision_final_artifacts.zip` containing

```
models/mobilenet_v2_best.pt
models/custom_cnn_best.pt
models/model_manifest.json
evaluation/final/metrics.json
evaluation/final/classification_report_*.txt
evaluation/final/confusion_matrix_*.png
evaluation/final/training_curves_*.png
evaluation/final/comparison.md
evaluation/final/comparison.png
evaluation/final/gradcam_samples/*.png
```

Install it locally with `python ml/scripts/install_artifacts.py <zip>`. The API and UI then switch from the development smoke checkpoint to the final model automatically.

> **Methodology rule:** the test split is evaluated exactly once, after all training and model selection is finished. Never tune on it. Validation macro-F1 drives checkpoint selection.

## 1. Check the GPU runtime

`Runtime → Change runtime type → T4 GPU` (or better). Training on CPU is possible but very slow.

In [ ]:
import platform
import torch

print("Python:", platform.python_version())
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")
else:
    raise SystemExit("No GPU detected. Set Runtime -> Change runtime type -> T4 GPU.")

## 2. Get the project into Colab

**Option A (recommended):** upload a zip of the project folder (without `.venv`, `ml/datasets`, trained weights).

**Option B:** clone your own Git remote by setting `REPO_URL` below.

In [ ]:
import os
import shutil
import zipfile
from pathlib import Path

PROJECT = Path("/content/ecovision-ai")
REPO_URL = ""  # e.g. "https://github.com/you/ecovision-ai.git"

if not (PROJECT / "ml" / "scripts" / "train_mobilenet.py").exists():
    if REPO_URL:
        !git clone {REPO_URL} {PROJECT}
    else:
        from google.colab import files

        uploaded = files.upload()  # choose ecovision-ai.zip
        archive = next(iter(uploaded))
        shutil.rmtree(PROJECT, ignore_errors=True)
        with zipfile.ZipFile(archive) as zf:
            zf.extractall("/content")
        if not PROJECT.exists():
            candidates = [p for p in Path("/content").iterdir() if (p / "ml" / "scripts").is_dir()]
            if not candidates:
                raise SystemExit("Could not find the project folder after extraction.")
            shutil.move(str(candidates[0]), str(PROJECT))

os.chdir(PROJECT)
print("Project root:", Path.cwd())
!ls

In [ ]:
# Colab already ships torch/torchvision. Install only the extra training stack.
%pip install -q "opencv-python-headless>=4.10,<5" "scikit-learn>=1.5,<2" "matplotlib>=3.9,<4" "pillow>=10.4,<13"

## 3. Download and prepare RealWaste

The Hugging Face mirror is used by default because it downloads per file and resumes cleanly. The UCI archive is the canonical source; pass `--source uci` if you prefer it.

Preparation computes SHA-256 and perceptual hashes, groups near-duplicates, and creates a leakage-safe 70/15/15 split (seed 42). It also caches 256 px JPEGs.

In [ ]:
!python ml/scripts/download_dataset.py --source hf

In [ ]:
!python ml/scripts/prepare_dataset.py

## 4. Train the models

**MobileNetV2 (primary model):** 4 epochs with the backbone frozen, then 12 epochs fine-tuning the last three inverted-residual blocks with a lower learning rate. Checkpoint selection uses validation macro-F1.

Expected runtime on a T4: roughly 25–40 minutes.

In [ ]:
!python ml/scripts/train_mobilenet.py --stage-a-epochs 4 --stage-b-epochs 12 --device cuda

**Custom CNN baseline:** up to 30 epochs, early stopping on validation macro-F1.

Expected runtime on a T4: roughly 15–25 minutes.

In [ ]:
!python ml/scripts/train_cnn.py --epochs 30 --device cuda

## 5. Evaluate once on the held-out test split

Produces accuracy, macro/weighted precision/recall/F1, per-class metrics, classification reports, confusion matrices, training curves, CPU latency, and the comparison table.

In [ ]:
!python ml/scripts/evaluate.py --stage final --device cuda

In [ ]:
import json
from pathlib import Path

metrics = json.loads(Path("ml/evaluation/final/metrics.json").read_text())
print(f"Test images: {metrics['test_split']['images']}")
print(f"Split hash:  {metrics['test_split']['split_hash'][:16]}...")
print()
print(f"{'Model':<14}{'Accuracy':>10}{'Macro F1':>10}{'Params':>12}")
for arch, entry in metrics["models"].items():
    data = entry["metrics"]
    print(f"{arch:<14}{data['accuracy']:>10.4f}{data['macro_f1']:>10.4f}{entry['parameters']:>12,}")

## 6. Generate Grad-CAM samples

Two test images per class, saved as three-panel figures (original / heatmap / overlay) for the README and model card.

In [ ]:
!python ml/scripts/gradcam.py --samples-per-class 2 --split test --output ml/evaluation/final/gradcam_samples

## 7. Package the artifacts

The zip is validated locally by `install_artifacts.py` before anything is copied into place.

In [ ]:
import shutil
from pathlib import Path

root = Path("ml")
staging = Path("/content/final_artifacts")
shutil.rmtree(staging, ignore_errors=True)
(staging / "models").mkdir(parents=True)
(staging / "evaluation" / "final").mkdir(parents=True)

for name in ("mobilenet_v2_best.pt", "custom_cnn_best.pt", "model_manifest.json"):
    shutil.copy2(root / "models" / name, staging / "models" / name)
shutil.copytree(root / "evaluation" / "final", staging / "evaluation" / "final", dirs_exist_ok=True)

archive = shutil.make_archive("/content/ecovision_final_artifacts", "zip", staging)
print("Created", archive)
!ls -lh /content/ecovision_final_artifacts.zip

In [ ]:
from google.colab import files

files.download("/content/ecovision_final_artifacts.zip")

## 8. Install locally

```bash
# from the project root
python ml/scripts/install_artifacts.py ~/Downloads/ecovision_final_artifacts.zip
# restart the API so it loads the final checkpoint
```

The installer checks that both checkpoints declare `stage: final`, that class order matches the app, that the manifest points at an installed model, and that `metrics.json` contains both models and the comparison table.

After installation, `/api/model-info` reports `is_final: true`, the UI development banner disappears, and the About page shows the final held-out test metrics.

**If this notebook produced poor results:** record them honestly. Do not hand-edit `metrics.json` or re-evaluate until the numbers improve. Adjust augmentation, class weights, or unfreezing strategy, retrain, and evaluate once more on the same untouched test split — noting how many configurations were tried.